## Importacion de Librerias - Modelo COMPLETO

In [2]:
# Framework principal de PyTorch para cálculo tensorial y Deep Learning.
import torch

# Módulo de PyTorch con capas, bloques y herramientas para construir redes neuronales.
import torch.nn as nn

# Funciones sin estado para redes neuronales (funciones de activación, pérdidas, etc.).
import torch.nn.functional as F

# use GPU
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

Vamos a construir el modelo GPT completo desde cero. Aunque en este punto no sea directamente usable para generar texto coherente porque contiene pesos inicializados aleatoriamente, implementarlo nos permite integrar todas las piezas que conforman la arquitectura Transformer en un solo sistema funcional.

---

### 1. El truco de eficiencia: Proyección combinada $QKV$

En implementaciones ingenuas, se definen tres capas lineales separadas para calcular las consultas, claves y valores:


$$W_Q, W_K, W_V \in \mathbb{R}^{d_{\text{embed}} \times d_{\text{embed}}}$$

Dado que estas tres matrices comparten exactamente las mismas dimensiones de entrada y tipo de datos, el estándar en frameworks como PyTorch consiste en **fusionarlas en una sola matriz de proyección lineal grande**:

$$W_{QKV} \in \mathbb{R}^{d_{\text{embed}} \times 3d_{\text{embed}}}$$

* **Ventaja computacional:** En lugar de lanzar tres operaciones de producto matricial (GEMM) independientes a la GPU, ejecutamos una sola operación de multiplicación lineal grande, aprovechando mucho mejor el ancho de banda y la memoria.
* **Separación (*Split*):** Tras proyectar la entrada, dividimos el tensor resultante en 3 partes iguales a lo largo del último eje usando funciones nativas de PyTorch como `torch.split()` o `.chunk()`.

---

### 2. Estructura modular en 3 clases de PyTorch

Para mantener el código desacoplado, legible y mantenible, estructuramos el modelo en tres clases principales:

1. **`MultiHeadAttention`:** Contiene la proyección unificada $QKV$, la división en múltiples cabezas, el escalado, la máscara causal, el cálculo de Softmax y la proyección final $W_O$.
2. **`TransformerBlock`:** Integra la subcapa de atención (`MultiHeadAttention`) y la subcapa `MLP` (Feed-Forward con expansión $4\times$, activación no lineal y contracción), aplicando las normalizaciones `LayerNorm` y las conexiones residuales.
3. **`GPTLanguageModel`:** Ensambla los embeddings de tokens y de posición, apila múltiples bloques `TransformerBlock` idénticos secuencialmente, y aplica la capa final de `LayerNorm` y el proyector de des-incrustación (*unembedding head*) para generar los logits sobre el vocabulario.

---

### 3. Conteo de parámetros entrenables (*Model Summary*)

Al finalizar la construcción del modelo, calculamos el número total de parámetros entrenables iterando sobre `model.parameters()`:

$$\text{Total Parameters} = \sum_{p \in \Theta} \text{numel}(p)$$

Esto nos permite verificar empíricamente que la arquitectura coincide con el presupuesto de parámetros teórico (por ejemplo, ~124M en la configuración de GPT-2 Small).

---

## Hyperparametros

Con lo que respecta a los modelos modernos estos pesos son insignificantes

In [5]:
# Hiperparámetros para GPT2-124M
n_vocab    = 50257  # Tamaño del vocabulario de GPT2
embed_dim  =   768  # Dimensión del embedding
seq_len    =  1024  # Longitud máxima de secuencia
n_heads    =    12  # Cabezales de atención que cada bloque tiene 12 y todas caben en 768 dimensiones
n_blocks   =    12  # Bloques de transformer
batch_size =     8

## Clase para Multi-Head Attention

In [4]:
class MultiHeadAttention(nn.Module):
  def __init__(self):
    super().__init__()

    # Número de cabezales de atención
    self.num_heads = n_heads
    self.head_dim  = embed_dim // n_heads

    # Las tres matrices de pesos Q, K, V se inicializan como una sola y se dividen dentro de forward()
    self.QKV = nn.Linear(embed_dim, 3*embed_dim, bias=True)

    # Mezcla lineal posterior a la atención
    self.W0 = nn.Linear(embed_dim, embed_dim, bias=True)


  def forward(self, x):

    # Dimensiones para usar más adelante
    B, T, E = x.shape # [batch, long_secuencia, dim_embedding]

    # Pasar los datos a través de Q, K y V en una sola matriz concatenada
    qkv = self.QKV(x) # [batch, secuencia, 3*dim_embedding]
    q, k, v = torch.split(qkv, E, dim=2) # Cada matriz es [B, T, E]

    # Redimensionar a [B, T, nHeads, head_dim]
    #  y luego transponer a [B, nHeads, T, head_dim]
    q = q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2) # [B, nHeads, T, head_dim]
    k = k.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
    v = v.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

    # La función de atención por producto punto escalado de PyTorch maneja dimensiones multicabezal
    out = F.scaled_dot_product_attention(q, k, v, is_causal=True) # [B, nHeads, T, head_dim]

    # Recombinar cabezales: (B, nHeads, T, head_dim) -> [B, T, E]
    out = out.transpose(1, 2).view(B, T, E)

    # Finalmente, proyectar linealmente los cabezales de atención
    out = self.W0(out)

    return out

## Clase para el bloque de Transformer

In [7]:
class TransformerBlock(nn.Module):
  def __init__(self):
    super().__init__()

    ### Subbloque de atención
    self.layernorm_1 = nn.LayerNorm(embed_dim, eps=1e-5)
    self.attn = MultiHeadAttention()

    ### Subbloque feedforward lineal (MLP)
    self.layernorm_2 = nn.LayerNorm(embed_dim, eps=1e-5)
    # Expansión 4x, luego de regreso al tamaño de embedding
    self.mlp_1 = nn.Linear(embed_dim, 4*embed_dim, bias=True)
    self.gelu  = nn.GELU()
    self.mlp_2 = nn.Linear(4*embed_dim, embed_dim, bias=True)

  def forward(self, x):

    # Atención
    x_att = self.layernorm_1(x) # Normalización previa a la atención
    x_att = x + self.attn(x_att) # Pasar por la atención, luego sumar la activación previa a la atención ("residual")

    # MLP
    x_ff = self.layernorm_2(x_att) # Normalización previa al MLP
    x_ff = x_att + self.mlp_2(self.gelu( self.mlp_1(x_ff) )) # Ajuste de la expansión-contracción

    return x_ff

## Y una clase para el modelo completo

In [23]:
class LanguageModel(nn.Module):
  def __init__(self):
    super().__init__()

    # Embeddings de tokens + posición
    self.wte = nn.Embedding(n_vocab, embed_dim) # Embedding de tokens
    self.wpe = nn.Embedding(seq_len, embed_dim) # Embedding de posición

    # Bloques de transformer
    self.transformerBlocks = nn.Sequential(*[TransformerBlock() for _ in range(n_blocks)])

    # LayerNorm final
    self.layernorm_final = nn.LayerNorm(embed_dim, eps=1e-5)

    # LM Head, con pesos vinculados al embedding de tokens (weight tying)
    self.final_head = nn.Linear(embed_dim, n_vocab, bias=False)
    self.final_head.weight = nn.Parameter(self.wte.weight) # (lo que sale al final en el summary)


  def forward(self, idx):

    # Embeddings de tokens + posición (¡ojo con el Device de la GPU)
    token_emb = self.wte(idx) # [B, T, E]
    posit_emb = self.wpe(torch.arange(idx.shape[-1], device=device)) # [T, E]
    x = token_emb + posit_emb # [B, T, E]

    # Pasar a través de cada bloque transformer
    x = self.transformerBlocks(x)

    # LayerNorm final y desproyección (unembedding)
    x = self.layernorm_final(x)
    logits = self.final_head(x)  # [B, T, n_vocab]

    return logits


  def generate(self, idx, temperature=1., max_new_tokens=50):

    for _ in range(max_new_tokens):

      # Pase hacia adelante (forward pass)
      logits = self(idx[:, -seq_len:])  # [B, T, n_vocab]
      logits = logits[:, -1, :]  # Logits del último token: [B, n_vocab]

      # Aplicar temperatura + softmax
      probs = F.softmax(logits / temperature, dim=-1) # [B, n_vocab]

      # Muestrear el siguiente token
      idx_next = torch.multinomial(probs, num_samples=1) # [B, 1]

      # Concatenar
      idx = torch.cat((idx, idx_next), dim=1) # [B, T+1]
    return idx

## Crear una instancia y probar el modelo

In [10]:
model = LanguageModel().to(device)
model

LanguageModel(
  (wte): Embedding(50257, 768)
  (wpe): Embedding(1024, 768)
  (transformerBlocks): Sequential(
    (0): TransformerBlock(
      (layernorm_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (attn): MultiHeadAttention(
        (QKV): Linear(in_features=768, out_features=2304, bias=True)
        (W0): Linear(in_features=768, out_features=768, bias=True)
      )
      (layernorm_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (mlp_1): Linear(in_features=768, out_features=3072, bias=True)
      (gelu): GELU(approximate='none')
      (mlp_2): Linear(in_features=3072, out_features=768, bias=True)
    )
    (1): TransformerBlock(
      (layernorm_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (attn): MultiHeadAttention(
        (QKV): Linear(in_features=768, out_features=2304, bias=True)
        (W0): Linear(in_features=768, out_features=768, bias=True)
      )
      (layernorm_2): LayerNorm((768,), eps=1e-05, elementwise_affine=Tr

In [11]:
# Pasar datos ficticios por el modelo
data = torch.randint(0, n_vocab, size=(batch_size, seq_len)).to(device)
out = model(data)

print(f'Tamaño de entrada:  {data.shape}')
print(f'Tamaño de salida:   {out.shape}')

Tamaño de entrada:  torch.Size([8, 1024])
Tamaño de salida:   torch.Size([8, 1024, 50257])


In [17]:
# La idea es que cada uno de los numeros del n_vocab sea un Logit para cada token en el vocabulario que despues se converte a probabilidades y la prediccion del siguiente token es la mas alta de esta distribucion (50257)

In [18]:
!pip install torchinfo # No está instalado por defecto
from torchinfo import summary

# Resumen del modelo y sus parámetros
summary(model, input_data=data, col_names=['input_size', 'output_size', 'num_params'])

Layer (type:depth-idx)                   Input Shape               Output Shape              Param #
LanguageModel                            [8, 1024]                 [8, 1024, 50257]          --
├─Embedding: 1-1                         [8, 1024]                 [8, 1024, 768]            38,597,376
├─Embedding: 1-2                         [1024]                    [1024, 768]               786,432
├─Sequential: 1-3                        [8, 1024, 768]            [8, 1024, 768]            --
│    └─TransformerBlock: 2-1             [8, 1024, 768]            [8, 1024, 768]            --
│    │    └─LayerNorm: 3-1               [8, 1024, 768]            [8, 1024, 768]            1,536
│    │    └─MultiHeadAttention: 3-2      [8, 1024, 768]            [8, 1024, 768]            2,362,368
│    │    └─LayerNorm: 3-3               [8, 1024, 768]            [8, 1024, 768]            1,536
│    │    └─Linear: 3-4                  [8, 1024, 768]            [8, 1024, 3072]           2,362,368
│ 

In [22]:
# Lo que no sabe la funcion Summary es que hemos unido las dos matrices de Embedings los iniciales y la desproyección

In [25]:
print(f'Total De Parametros entrenables reales = {163037184 - 38597376:,}')

Total De Parametros entrenables reales = 124,439,808


Los modelos comerciales de frontera son, en su esencia fundamental, la misma arquitectura: operan con matrices notablemente más grandes, apilan decenas de capas, se entrenan con conjuntos masivos de datos y escalan el cómputo. Si inspeccionaras el código base de los modelos más avanzados de la familia GPT, verías una estructura modular con clases, tensores, proyecciones y flujos residuales prácticamente idénticos a los que acabamos de construir.

Sin embargo, **existe una gran excepción en cómo ha evolucionado el diseño arquitectónico moderno**:

---

### 1. El salto de Dense a MoE (Mixture of Experts)

Mientras que un modelo clásico es **denso** (cada token activa el 100% de los parámetros y de la subcapa MLP en cada capa), los modelos comerciales de gran escala adoptaron arquitecturas **MoE (Mixture of Experts)**:

* La subcapa MLP monolítica se reemplaza por múltiples sub-redes expertas independientes (por ejemplo, 8 o 16 expertos por bloque).
* Un mecanismo de enrutamiento (*Router/Gating Network*) selecciona dinámicamente solo a los mejores expertos (por ejemplo, los *Top-2*) para procesar cada token.
* **Impacto:** Permite tener cientos de miles de millones de parámetros totales en memoria, pero ejecutando solo una fracción de ellos por token, abaratando drásticamente el costo de inferencia y aumentando el *throughput*.

---

### 2. Optimización de Memoria y Contexto Masivo

Los modelos actuales no procesan el contexto como el GPT clásico:

* **Grouped-Query Attention (GQA):** Reduce la huella de memoria del *KV-Cache* durante la generación larga al compartir claves y valores entre múltiples cabezas de consulta.
* **Rotary Position Embeddings (RoPE) y Context Windows extendidas:** Reemplazaron los embeddings posicionales fijos para permitir ventanas de contexto que escalan desde 32k hasta millones de tokens.
* **FlashAttention:** A nivel de kernel de bajo nivel en GPU, la atención ya no materializa matrices gigantes en memoria global (VRAM), sino que calcula el Softmax por bloques directamente en la memoria SRAM ultrarrápida del chip.

---

### 3. Estabilidad Numérica y Eficiencia de Parámetros

* **RMSNorm en lugar de LayerNorm:** Elimina el cálculo del centrado de la media para reducir la latencia de cómputo en hardware.
* **SwiGLU / GeGLU:** Sustituyen a la activación GELU estándar por compuertas multiplicativas que incrementan la capacidad de representación en el MLP.
* **Eliminación de sesgos (`bias=False`):** En casi todas las proyecciones lineales para ahorrar memoria y mejorar la estabilidad en regímenes de entrenamiento con precisión mixta (FP16/BF16/FP8).

El esqueleto matemático y el flujo de autoatención autorregresiva siguen intactos, pero la ingeniería alrededor de la eficiencia de memoria, el enrutamiento disperso y el aprovechamiento del hardware es lo que define a los modelos de frontera actuales.